# Build CLAY's per-attribute prompt bank

This notebook produces a single artifact: **`clip_attr_prompt_bank.pt`** (shape `[40, n, 512]`).

**Why it exists.** Tier-0 uses one text vector per attribute. CLAY does not — it builds a *subspace* per attribute by running SVD on a stack of many prompts that all describe the same attribute (`CLAY.md` §3.2). With one prompt that stack is rank-1 and CLAY is undefined. This notebook generates ~12–24 deterministic, template-based prompts per attribute, encodes them with frozen CLIP ViT-B/32, and caches the stack so CLAY's `log-map → SVD → P_c = V_k V_kᵀ` has real material to work on.

**Run order:** top to bottom. Set the runtime to GPU (Runtime → Change runtime type → GPU) — though for 690 short text prompts CPU is also fine.

Output: `Deep-Learning-Project/artifacts/clip_attr_prompt_bank.pt`, optionally copied to Google Drive so it survives a runtime reset.

## 1. Install dependencies

In [ ]:
!pip install -q transformers torch torchvision

## 2. Get the code

Clone the repo. If it already exists (you re-ran the cell), pull the latest instead of failing.

In [ ]:
import os

REPO_URL = "https://github.com/davideCabitz/Deep-Learning-Project.git"
REPO_DIR = "/content/Deep-Learning-Project"

if not os.path.isdir(REPO_DIR):
    !git clone $REPO_URL $REPO_DIR
else:
    !cd $REPO_DIR && git pull

# Put src/ on the path so `import clip_prompts` works (the modules use flat imports).
import sys
SRC_DIR = os.path.join(REPO_DIR, "src")
if SRC_DIR not in sys.path:
    sys.path.insert(0, SRC_DIR)

print("src on path:", SRC_DIR)
print("files:", os.listdir(SRC_DIR))

## 3. (Optional) Preview the prompts before encoding

Sanity-check the generated prompts — pure Python, no CLIP needed. Confirms every attribute has `n>1` prompts (the whole point) and they read naturally.

In [ ]:
from clip_prompts import build_prompts_for_attribute, _verify_coverage
from data_loader import ATTRIBUTE_NAMES

_verify_coverage()  # all 40 attrs mapped, no extras

counts = {n: len(build_prompts_for_attribute(n)) for n in ATTRIBUTE_NAMES}
print(f"prompts/attr: min={min(counts.values())}  max={max(counts.values())}  total={sum(counts.values())}")

for name in ["Eyeglasses", "Young", "Heavy_Makeup", "No_Beard"]:
    ps = build_prompts_for_attribute(name)
    print(f"\n=== {name}  ({len(ps)} prompts) ===")
    for p in ps[:5]:
        print("   ", p)

## 4. Build the prompt bank

Encodes every prompt with frozen CLIP ViT-B/32 and caches `[40, n, 512]` to `artifacts/`. First run downloads the CLIP weights (~600 MB). Pass `force=True` to rebuild after editing the synonym tables.

In [ ]:
from clip_prompts import extract_prompt_bank

extract_prompt_bank(force=False)

## 5. Verify the artifact

Independent check of what landed on disk: shape `[40, n, 512]`, every row unit-normalized, and `n>1` so CLAY's subspace is non-trivial.

In [ ]:
import torch
from data_loader import _get_artifacts_dir

path = _get_artifacts_dir() / "clip_attr_prompt_bank.pt"
bank = torch.load(path)

print("path :", path)
print("shape:", tuple(bank.shape), "  (attrs, prompts_per_attr, dim)")
print("dtype:", bank.dtype)
norms = bank.norm(dim=2)
print("row norms: min=%.5f max=%.5f  (expect ~1.0)" % (norms.min(), norms.max()))
assert bank.shape[0] == 40 and bank.shape[2] == 512 and bank.shape[1] > 1
assert torch.allclose(norms, torch.ones_like(norms), atol=1e-4)
print("\n[OK] prompt bank is valid — CLAY can build per-attribute subspaces from it.")

## 6. Save to Google Drive (so it survives a runtime reset)

The Colab disk is wiped when the runtime recycles. Copy the artifact to Drive to keep it. Skip this cell if you only need it for the current session.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

DRIVE_DIR = "/content/drive/MyDrive/Deep-Learning-Project/artifacts"
os.makedirs(DRIVE_DIR, exist_ok=True)
!cp "$path" "$DRIVE_DIR/"
print("copied to:", DRIVE_DIR)
!ls -lh "$DRIVE_DIR"